<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Supplementary_TMS_fMRI_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Supplementary_TMS_fMRI_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validation Controls for FC Analysis

**Objective:** Validate that the FC correlations reported in `Test_TMS_Models_on_RestingState.ipynb` are not trivial.

**Approach:**
1. **Empirical baseline**: Compute correlation between empirical rest and empirical stim FC structures
2. **Ablation study**: Train models WITHOUT stimulus input channel and compare FC accuracy
3. **Region-level analysis**: Identify which brain regions/connections the model predicts best

**Questions to answer:**
- Is Rest FC correlation (0.67) better than just the natural rest↔stim correlation?
- Does the stimulus input channel actually improve FC predictions?
- Are stimulus-target regions predicted better than non-targets?

## Step 1: Setup and Load Data

In [1]:
# --- Setup ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, subprocess

repo_dir = "/content/BrainStim_ANN_fMRI_HCP"
if not os.path.exists(repo_dir):
    !git clone https://github.com/grabuffo/BrainStim_ANN_fMRI_HCP.git
else:
    print("Repo already exists ✅")

# Define paths
base_dir = "/content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data"
data_dir = os.path.join(base_dir, "TMS_fMRI")
preproc_dir = os.path.join(base_dir, "preprocessed_subjects_tms_fmri")
dataset_pkl = os.path.join(data_dir, "dataset_tian50_schaefer400_allruns.pkl")

weights_dir = os.path.join(preproc_dir, "trained_models_MLP_tms_fmri_persubject")
results_dir = os.path.join(preproc_dir, "results_tms_fmri_persubject")
test_results_dir = os.path.join(preproc_dir, "results_tms_models_on_rest")
validation_dir = os.path.join(preproc_dir, "validation_fc_controls")
os.makedirs(validation_dir, exist_ok=True)

sys.path.append(repo_dir)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pickle
import json
from src.preprocessing_tms_fmri import preprocess_run

import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load dataset
with open(dataset_pkl, "rb") as f:
    dataset_emp = pickle.load(f)

print(f"✅ Loaded dataset with {len(dataset_emp)} subjects")

Mounted at /content/drive
Cloning into 'BrainStim_ANN_fMRI_HCP'...
remote: Enumerating objects: 694, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 694 (delta 74), reused 14 (delta 14), pack-reused 538 (from 1)
Receiving objects: 100% (694/694), 88.47 MiB | 17.06 MiB/s, done.
Resolving deltas: 100% (251/251), done.
Device: cuda
✅ Loaded dataset with 46 subjects


## Step 2: Configuration & Load Models

In [3]:
# --- Configuration (match Test_TMS_Models_on_RestingState.ipynb) ---
ROI_num = 450
using_steps = 3
tr_stim = 2.4
tr_rest = 2.0
remove_initial_trs = 30
low_hz, high_hz = 0.008, 0.08

# Model class WITH stimulus
class ANN_MLP_with_Stimulus(nn.Module):
    def __init__(self, roi_num, using_steps, hidden_dim=256, latent_dim=128):
        super().__init__()
        brain_dim = using_steps * roi_num
        stim_dim = roi_num
        stim_timing_dim = 1
        input_dim = brain_dim + stim_dim + stim_timing_dim
        output_dim = roi_num

        self.func = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, output_dim),
        )

    def forward(self, x):
        return self.func(x)

# Model class WITHOUT stimulus (for ablation)
class ANN_MLP_no_Stimulus(nn.Module):
    def __init__(self, roi_num, using_steps, hidden_dim=256, latent_dim=128):
        super().__init__()
        brain_dim = using_steps * roi_num
        input_dim = brain_dim  # NO stimulus info
        output_dim = roi_num

        self.func = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, output_dim),
        )

    def forward(self, x):
        return self.func(x)

print("Model architectures defined (WITH and WITHOUT stimulus)")

Model architectures defined (WITH and WITHOUT stimulus)


In [4]:
# --- Load trained models WITH stimulus ---
trained_models_with_stim = {}

model_files = [f for f in os.listdir(weights_dir) if f.endswith('_MLP_with_stim.pt')]
print(f"Found {len(model_files)} trained models")

loaded_count = 0
for model_file in sorted(model_files)[:5]:  # Load first 5 for speed
    sub_id = model_file.replace('_MLP_with_stim.pt', '')
    model_path = os.path.join(weights_dir, model_file)

    try:
        model = ANN_MLP_with_Stimulus(roi_num=ROI_num, using_steps=using_steps)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval()
        trained_models_with_stim[sub_id] = model
        loaded_count += 1
        print(f"✓ {sub_id}")
    except RuntimeError as e:
        if "size mismatch" in str(e):
            print(f"✗ {sub_id} - skipping (old model)")
        else:
            raise

print(f"\n✅ Loaded {loaded_count} models WITH stimulus")

Found 40 trained models
✓ sub-NTHC1015
✓ sub-NTHC1016
✓ sub-NTHC1019
✓ sub-NTHC1021
✓ sub-NTHC1022

✅ Loaded 5 models WITH stimulus


## Step 3: Compute Empirical Baseline - Rest↔Stim FC Correlation

In [6]:
# --- Utility function ---
def compute_fc_matrix(time_series):
    """Compute functional connectivity (Pearson correlation) matrix."""
    ts_std = (time_series - time_series.mean(axis=0)) / (time_series.std(axis=0) + 1e-8)
    fc = np.corrcoef(ts_std.T)
    return fc

def extract_upper_triangle(fc_matrix):
    """Extract upper triangle of FC (excluding diagonal) as vector."""
    N = fc_matrix.shape[0]
    indices = np.triu_indices(N, k=1)
    return fc_matrix[indices]

print("Utility functions defined")

Utility functions defined


In [7]:
# --- CONTROL 1: Empirical Rest ↔ Stim FC correlation ---
# Question: How correlated are rest and stim FC to BEGIN WITH?

print("="*70)
print("CONTROL 1: EMPIRICAL REST ↔ STIM FC CORRELATION")
print("(Baseline: natural correlation between rest and stim network structures)")
print("="*70)

empirical_baseline = {}

for sub_id in sorted(trained_models_with_stim.keys()):
    print(f"\n{sub_id}:")

    if 'task-stim' not in dataset_emp[sub_id] or 'task-rest' not in dataset_emp[sub_id]:
        print("  Missing data")
        continue

    rest_ts_list = []
    stim_ts_list = []

    # Collect rest time series
    for sess_idx in sorted(dataset_emp[sub_id]['task-rest'].keys()):
        ts = dataset_emp[sub_id]['task-rest'][sess_idx].get('time series')
        if ts is None or len(ts) == 0:
            continue
        ts = np.asarray(ts, dtype=np.float32)
        if ts.shape[0] <= remove_initial_trs:
            continue
        ts_drop = ts[remove_initial_trs:, :]
        ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0,
                                 low=low_hz, high=high_hz, order=2, zscore=True)
        if ts_proc.shape[0] > using_steps:
            rest_ts_list.append(ts_proc)

    # Collect stim time series
    for sess_idx in sorted(dataset_emp[sub_id]['task-stim'].keys()):
        ts = dataset_emp[sub_id]['task-stim'][sess_idx].get('time series')
        if ts is None or len(ts) == 0:
            continue
        ts = np.asarray(ts, dtype=np.float32)
        if ts.shape[0] <= remove_initial_trs:
            continue
        ts_drop = ts[remove_initial_trs:, :]
        ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0,
                                 low=low_hz, high=high_hz, order=2, zscore=True)
        if ts_proc.shape[0] > using_steps:
            stim_ts_list.append(ts_proc)

    if rest_ts_list and stim_ts_list:
        rest_ts = np.vstack(rest_ts_list)
        stim_ts = np.vstack(stim_ts_list)

        # Compute empirical FC
        fc_rest_emp = compute_fc_matrix(rest_ts)
        fc_stim_emp = compute_fc_matrix(stim_ts)

        # Extract vectors
        fc_rest_vec = extract_upper_triangle(fc_rest_emp)
        fc_stim_vec = extract_upper_triangle(fc_stim_emp)

        # Compute correlation
        r_empirical = np.corrcoef(fc_rest_vec, fc_stim_vec)[0, 1]

        empirical_baseline[sub_id] = {
            'rest_fc': fc_rest_emp,
            'stim_fc': fc_stim_emp,
            'correlation': float(r_empirical),
        }

        print(f"  ✓ Empirical Rest ↔ Stim FC correlation: r={r_empirical:.3f}")

if empirical_baseline:
    corrs = [empirical_baseline[s]['correlation'] for s in empirical_baseline]
    print(f"\n" + "="*70)
    print(f"Mean empirical rest↔stim correlation: {np.mean(corrs):.3f} ± {np.std(corrs):.3f}")
    print(f"(This is the BASELINE. Model rest FC r=0.67 should be BETTER than this!)")
    print("="*70)

CONTROL 1: EMPIRICAL REST ↔ STIM FC CORRELATION
(Baseline: natural correlation between rest and stim network structures)

sub-NTHC1015:
  ✓ Empirical Rest ↔ Stim FC correlation: r=0.522

sub-NTHC1016:
  ✓ Empirical Rest ↔ Stim FC correlation: r=0.222

sub-NTHC1019:
  ✓ Empirical Rest ↔ Stim FC correlation: r=0.429

sub-NTHC1021:
  ✓ Empirical Rest ↔ Stim FC correlation: r=0.484

sub-NTHC1022:
  ✓ Empirical Rest ↔ Stim FC correlation: r=0.442

Mean empirical rest↔stim correlation: 0.420 ± 0.104
(This is the BASELINE. Model rest FC r=0.67 should be BETTER than this!)


## Step 4: Ablation Study - Models WITHOUT Stimulus Input

In [8]:
# --- CONTROL 2: Train models WITHOUT stimulus info on the fly ---
# (We'll use same data but zero out stimulus channels during inference)

print("="*70)
print("CONTROL 2: ABLATION - MODEL WITHOUT STIMULUS")
print("(Does stimulus input actually improve predictions?)")
print("="*70)

def get_model_prediction_no_stim(model_with_stim, inputs_with_stim):
    """
    Take model trained WITH stimulus, but remove stimulus channels from input.
    Input format: [brain_state (1350) | target (450) | stim_pulse (1)]
    Remove: target and stim_pulse, keep only brain_state (1350)
    """
    # Extract only brain state
    input_dim_with_stim = 1350 + 450 + 1  # 1801
    brain_state_only = inputs_with_stim[:, :1350]  # Keep first 1350

    # Zero pad the removed channels so model still works
    # (model expects 1801 input, give it [brain_state | zeros | zeros])
    inputs_no_stim = np.zeros_like(inputs_with_stim)
    inputs_no_stim[:, :1350] = brain_state_only

    # Forward pass
    with torch.no_grad():
        X_tensor = torch.tensor(inputs_no_stim, dtype=torch.float32, device=device)
        Y_pred = model_with_stim(X_tensor).cpu().numpy()

    return Y_pred

# Test ablation
ablation_results = {}

for sub_id in sorted(trained_models_with_stim.keys()):
    print(f"\n{sub_id}:")

    model = trained_models_with_stim[sub_id]
    model.eval()

    if 'task-rest' not in dataset_emp[sub_id]:
        continue

    rest_sessions = dataset_emp[sub_id]['task-rest']
    all_targets = []
    all_preds_with_stim = []
    all_preds_no_stim = []

    # Process rest sessions
    for sess_idx in sorted(rest_sessions.keys()):
        ts = rest_sessions[sess_idx].get('time series')
        if ts is None or len(ts) == 0:
            continue

        ts = np.asarray(ts, dtype=np.float32)
        if ts.shape[0] <= remove_initial_trs:
            continue

        ts_drop = ts[remove_initial_trs:, :]
        ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0,
                                 low=low_hz, high=high_hz, order=2, zscore=True)

        if ts_proc.shape[0] <= using_steps:
            continue

        T, N = ts_proc.shape

        # Build inputs (no stimulus, stub target)
        stub_target = np.zeros(ROI_num, dtype=np.float32)
        stim_pulse = np.zeros(T, dtype=np.float32)

        inputs = np.empty((T - using_steps, using_steps * N + N + 1), dtype=np.float32)
        for t in range(T - using_steps):
            brain_state = ts_proc[t : t + using_steps].reshape(-1)
            inputs[t] = np.concatenate([brain_state, stub_target, [stim_pulse[t]]])

        targets = ts_proc[using_steps:, :]

        # Predictions WITH stimulus input
        with torch.no_grad():
            X_tensor = torch.tensor(inputs, dtype=torch.float32, device=device)
            Y_pred_with = model(X_tensor).cpu().numpy()

        # Predictions WITHOUT stimulus input
        Y_pred_without = get_model_prediction_no_stim(model, inputs)

        all_targets.append(targets)
        all_preds_with_stim.append(Y_pred_with)
        all_preds_no_stim.append(Y_pred_without)

    if all_targets:
        targets = np.vstack(all_targets)
        preds_with = np.vstack(all_preds_with_stim)
        preds_without = np.vstack(all_preds_no_stim)

        # Compute FC for both
        fc_emp = compute_fc_matrix(targets)
        fc_with = compute_fc_matrix(preds_with)
        fc_without = compute_fc_matrix(preds_without)

        # Extract vectors
        fc_emp_vec = extract_upper_triangle(fc_emp)
        fc_with_vec = extract_upper_triangle(fc_with)
        fc_without_vec = extract_upper_triangle(fc_without)

        # Correlations
        r_with = np.corrcoef(fc_emp_vec, fc_with_vec)[0, 1]
        r_without = np.corrcoef(fc_emp_vec, fc_without_vec)[0, 1]
        delta = r_with - r_without

        ablation_results[sub_id] = {
            'fc_with_stim': float(r_with),
            'fc_without_stim': float(r_without),
            'delta': float(delta),
        }

        print(f"  WITH stimulus input:    r={r_with:.3f}")
        print(f"  WITHOUT stimulus input: r={r_without:.3f}")
        print(f"  Difference (Δ):        {delta:+.3f}  {'✓ stimulus helps!' if delta > 0.05 else '✗ stimulus minimal'}")

if ablation_results:
    deltas = [ablation_results[s]['delta'] for s in ablation_results]
    print(f"\n" + "="*70)
    print(f"Mean improvement from stimulus: {np.mean(deltas):+.3f}")
    print(f"(If < 0.03, stimulus input is NOT critical for FC prediction)")
    print("="*70)

CONTROL 2: ABLATION - MODEL WITHOUT STIMULUS
(Does stimulus input actually improve predictions?)

sub-NTHC1015:
  WITH stimulus input:    r=0.644
  WITHOUT stimulus input: r=0.644
  Difference (Δ):        +0.000  ✗ stimulus minimal

sub-NTHC1016:
  WITH stimulus input:    r=0.553
  WITHOUT stimulus input: r=0.553
  Difference (Δ):        +0.000  ✗ stimulus minimal

sub-NTHC1019:
  WITH stimulus input:    r=0.566
  WITHOUT stimulus input: r=0.566
  Difference (Δ):        +0.000  ✗ stimulus minimal

sub-NTHC1021:
  WITH stimulus input:    r=0.698
  WITHOUT stimulus input: r=0.698
  Difference (Δ):        +0.000  ✗ stimulus minimal

sub-NTHC1022:
  WITH stimulus input:    r=0.606
  WITHOUT stimulus input: r=0.606
  Difference (Δ):        +0.000  ✗ stimulus minimal

Mean improvement from stimulus: +0.000
(If < 0.03, stimulus input is NOT critical for FC prediction)


## Step 5: Region-Level Analysis - Which Connections Predict Best?

In [9]:
# --- CONTROL 3: Region-level FC correlation ---
# Which brain region pairs does the model predict best?
# If stimulus-target regions are predicted better → model learned selectivity

print("="*70)
print("CONTROL 3: REGION-LEVEL ANALYSIS")
print("(Which connections does the model predict best?)")
print("="*70)

region_analysis = {}

example_sub = sorted(trained_models_with_stim.keys())[0]
print(f"\nDetailed analysis for: {example_sub}")

model = trained_models_with_stim[example_sub]
model.eval()

# Get stim session data
if 'task-stim' in dataset_emp[example_sub]:
    stim_sessions = dataset_emp[example_sub]['task-stim']

    stim_ts_list = []
    stim_preds_list = []
    target_vec_example = None

    for sess_idx in sorted(stim_sessions.keys()):
        sess_data = stim_sessions[sess_idx]
        ts = sess_data.get('time series')
        target_vec = sess_data.get('target')
        stim_time_df = sess_data.get('stim time')

        if ts is None or target_vec is None:
            continue

        ts = np.asarray(ts, dtype=np.float32)
        if ts.shape[0] <= remove_initial_trs:
            continue

        ts_drop = ts[remove_initial_trs:, :]
        ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0,
                                 low=low_hz, high=high_hz, order=2, zscore=True)

        if ts_proc.shape[0] <= using_steps:
            continue

        target_vec = np.asarray(target_vec, dtype=np.float32).ravel()
        target_vec_example = target_vec

        # Extract stimulus timing
        T, N = ts_proc.shape
        stim_pulse = np.zeros(T, dtype=np.float32)
        if stim_time_df is not None and len(stim_time_df) > 0:
            stim_onsets = stim_time_df['onset'].values
            raw_tr_indices = np.round(stim_onsets / tr_stim).astype(int)
            valid_raw_indices = raw_tr_indices[raw_tr_indices >= remove_initial_trs]
            preprocessed_indices = valid_raw_indices - remove_initial_trs
            valid_preprocessed_indices = preprocessed_indices[preprocessed_indices < ts_proc.shape[0]]
            stim_pulse[valid_preprocessed_indices] = 1.0

        # Build inputs
        stim_inputs = np.empty((T - using_steps, using_steps * N + N + 1), dtype=np.float32)
        for t in range(T - using_steps):
            brain_state = ts_proc[t : t + using_steps].reshape(-1)
            stim_inputs[t] = np.concatenate([brain_state, target_vec, [stim_pulse[t]]])

        stim_targets = ts_proc[using_steps:, :]

        # Predictions
        with torch.no_grad():
            X_tensor = torch.tensor(stim_inputs, dtype=torch.float32, device=device)
            Y_pred = model(X_tensor).cpu().numpy()

        stim_ts_list.append(stim_targets)
        stim_preds_list.append(Y_pred)

    if stim_ts_list and target_vec_example is not None:
        stim_ts = np.vstack(stim_ts_list)
        stim_preds = np.vstack(stim_preds_list)

        # Find target and non-target regions
        target_region_idx = np.argmax(target_vec_example)
        target_mask = target_vec_example > 0.5
        non_target_mask = ~target_mask

        target_indices = np.where(target_mask)[0]
        non_target_indices = np.where(non_target_mask)[0]

        # Compute per-region prediction accuracy
        region_corrs = []
        for roi in range(ROI_num):
            if np.std(stim_ts[:, roi]) > 1e-6 and np.std(stim_preds[:, roi]) > 1e-6:
                r = np.corrcoef(stim_ts[:, roi], stim_preds[:, roi])[0, 1]
                if not np.isnan(r):
                    region_corrs.append((roi, r))

        # Split into target and non-target
        target_corrs = [r for roi, r in region_corrs if roi in target_indices]
        non_target_corrs = [r for roi, r in region_corrs if roi in non_target_indices]

        print(f"\nStimulation target region: ROI {target_region_idx}")
        print(f"\nTarget regions (n={len(target_indices)}):")
        print(f"  Mean correlation: {np.mean(target_corrs):.3f} ± {np.std(target_corrs):.3f}")
        print(f"\nNon-target regions (n={len(non_target_indices)}):")
        print(f"  Mean correlation: {np.mean(non_target_corrs):.3f} ± {np.std(non_target_corrs):.3f}")
        print(f"\nDifference (Target - Non-target): {np.mean(target_corrs) - np.mean(non_target_corrs):+.3f}")
        print(f"  {'✓ Target regions predicted better!' if np.mean(target_corrs) > np.mean(non_target_corrs) else '✗ No selectivity'}")

        region_analysis[example_sub] = {
            'target_region': int(target_region_idx),
            'target_region_corr': float(np.mean(target_corrs)),
            'non_target_region_corr': float(np.mean(non_target_corrs)),
            'selectivity_gap': float(np.mean(target_corrs) - np.mean(non_target_corrs)),
        }

CONTROL 3: REGION-LEVEL ANALYSIS
(Which connections does the model predict best?)

Detailed analysis for: sub-NTHC1015

Stimulation target region: ROI 231

Target regions (n=1):
  Mean correlation: 0.940 ± 0.000

Non-target regions (n=449):
  Mean correlation: 0.936 ± 0.023

Difference (Target - Non-target): +0.004
  ✓ Target regions predicted better!


## Step 6: Summary and Conclusions

In [10]:
# --- Summary table ---
print("\n" + "="*70)
print("VALIDATION SUMMARY")
print("="*70)

print("\n1. EMPIRICAL BASELINE (Rest↔Stim natural correlation):")
if empirical_baseline:
    baseline_corrs = [empirical_baseline[s]['correlation'] for s in empirical_baseline]
    print(f"   Mean: {np.mean(baseline_corrs):.3f} ± {np.std(baseline_corrs):.3f}")
    print(f"   Interpretation: Rest and Stim FC are naturally this correlated")
    print(f"   → Model's Rest FC (0.67) should EXCEED this baseline")

print("\n2. ABLATION STUDY (Stimulus input effect):")
if ablation_results:
    deltas = [ablation_results[s]['delta'] for s in ablation_results]
    print(f"   Mean improvement: {np.mean(deltas):+.3f} ± {np.std(deltas):.3f}")
    if np.mean(deltas) > 0.05:
        print(f"   ✓✓ STIMULUS INPUT IS CRITICAL (Δ > 0.05)")
    elif np.mean(deltas) > 0.02:
        print(f"   ✓ Stimulus input helps somewhat (Δ = 0.02-0.05)")
    else:
        print(f"   ✗ STIMULUS INPUT IS MINIMAL (Δ < 0.02) - potential triviality!")

print("\n3. REGION-LEVEL SELECTIVITY:")
if region_analysis:
    for sub_id in region_analysis:
        gap = region_analysis[sub_id]['selectivity_gap']
        print(f"   Target vs Non-target gap: {gap:+.3f}")
        if gap > 0.05:
            print(f"   ✓ Model shows SELECTIVITY for stimulation target")
        else:
            print(f"   ✗ No region-specific selectivity")

print("\n" + "="*70)


VALIDATION SUMMARY

1. EMPIRICAL BASELINE (Rest↔Stim natural correlation):
   Mean: 0.420 ± 0.104
   Interpretation: Rest and Stim FC are naturally this correlated
   → Model's Rest FC (0.67) should EXCEED this baseline

2. ABLATION STUDY (Stimulus input effect):
   Mean improvement: +0.000 ± 0.000
   ✗ STIMULUS INPUT IS MINIMAL (Δ < 0.02) - potential triviality!

3. REGION-LEVEL SELECTIVITY:
   Target vs Non-target gap: +0.004
   ✗ No region-specific selectivity



## Interpretation

**Three validation checks determine if FC analysis is trivial:**

1. **Empirical Baseline**: If model Rest FC (0.67) > empirical rest↔stim correlation, then it's NOT trivial
   - If baseline is 0.50 and model achieves 0.67 → ✓ real learning
   - If baseline is 0.80 and model achieves 0.67 → ✗ model underperforms

2. **Ablation (Stimulus Effect)**: Does removing stimulus hurt predictions?
   - Δ > 0.05 → ✓✓ stimulus is essential
   - Δ = 0.02-0.05 → ✓ stimulus helps moderately
   - Δ < 0.02 → ✗ stimulus is negligible (potential triviality)

3. **Region Selectivity**: Does model predict target regions better?
   - Gap > 0.05 → ✓ learned selective stimulus responses
   - Gap ≈ 0 → ✗ no specificity (just smoothing predictions)

**Conclusion**: If all three checks pass, FC analysis demonstrates real learned dynamics!